Reference:

[**Introduction to LangChain for Agentic AI**](https://courses.analyticsvidhya.com/courses/take/introduction-to-langchain-for-agentic-ai/lessons/61748853-course-introduction) 

## Install OpenAI, and LangChain dependencies

In [ ]:
# !pip install -qq langchain
# !pip install -qq langchain-openai
# !pip install -qq langchain-community
# !pip install -qq langchain-chroma

## Enter Open AI API Key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

## Load Connection to LLM

Here we create a connection to ChatGPT to use later in our chains

In [ ]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

## Multi-user Window-based Conversation Chains with persistence - SQLChatMessageHistory

The beauty of `SQLChatMessageHistory` is that we can store separate conversation histories per user or session which is often the need for real-world chatbots which will be accessed by many users at the same time. Instead of in-memory we can store it in a SQL database which can be used to store a lot of conversations.

We use a `get_session_history` function which is expected to take in a `session_id` and return a Message History object. Everything is stored in a SQL database. This `session_id` is used to distinguish between separate conversations, and should be passed in as part of the config when calling the new chain

We also use a `memory_buffer_window` function to only use the top-K last historical conversations before sending it to the LLM, basically our own implementation of `ConversationBufferWindowMemory`


- `ChatMessageHistory`: This is a class in LangChain that stores a sequence of chat messages (such as HumanMessage, AIMessage, SystemMessage) for a conversation. It allows you to append new messages and retrieve the full message history, which is useful for maintaining conversational context.

- `BaseChatMessageHistory`: This is an abstract base class that defines the interface for chat message history storage backends. It specifies methods like add_message, get_messages, and clear, which concrete implementations (like ChatMessageHistory, RedisChatMessageHistory, etc.) must provide.

- `RunnableWithMessageHistory`: This is a wrapper for a Runnable (such as a chain or LLM) that automatically manages message history for each session or user. It uses a function (like get_session_history) to retrieve the appropriate message history object based on a session ID, and injects the history into the chain's input. This enables multi-user or multi-session conversational experiences.

- `MessagesPlaceholder`: This is a special prompt template variable in LangChain that acts as a placeholder for a list of messages (e.g., the conversation history). When rendering a prompt, MessagesPlaceholder is replaced with the actual messages from history, allowing the LLM to see the full conversation context.


In [ ]:
# removes the memory database file - usually not needed
# you can run this only when you want to remove all conversation histories
# !rm memory.db

In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# used to retrieve conversation history from database
# based on a specific user or session ID
def get_session_history_db(session_id):
    return SQLChatMessageHistory(session_id, "sqlite:///memory.db")

# prompt to load in history and current input from the user
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "Act as a helpful AI Assistant"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{human_input}"),
    ]
)

# create a memory buffer window function to return the last K conversations
def memory_buffer_window(messages, k=2):
    return messages[-(k+1):]

# create a basic LLM Chain which only sends the last K conversations per user
llm_chain = (
    RunnablePassthrough.assign(history=lambda x: memory_buffer_window(x["history"]))
      |
    prompt_template
      |
    chatgpt
      | 
    StrOutputParser()
)

In [ ]:
# create a conversation chain which can load memory based on specific user or session id
conv_chain = RunnableWithMessageHistory(
    llm_chain,
    get_session_history_db,
    input_messages_key="human_input",
    history_messages_key="history",
)

# create a utility function to take in current user input prompt and their session ID
# streams result live back to the user from the LLM
def chat_with_llm(prompt: str, session_id: str):
    for chunk in conv_chain.stream({"human_input": prompt},
                                   {'configurable': { 'session_id': session_id}}):
        # print(chunk.content, end="")
        print(chunk, end="")

In [ ]:
user_id = 'jim001'
prompt = "Hi can you tell me which is the fastest animal?"
chat_with_llm(prompt, user_id)

In [ ]:
prompt = "what about the slowest animal?"
chat_with_llm(prompt, user_id)


#### ![Chat Message Memory Flow](../images/sqllite.png)

In [ ]:
prompt = "what about the largest animal?"
chat_with_llm(prompt, user_id)

In Below cell we will see the LLM always will get last 2 message in its history as we have kept window = 2

In [ ]:
prompt = "what topics have we discussed, show briefly as bullet points"
chat_with_llm(prompt, user_id)

In [ ]:
user_id = 'john005'
prompt = "Explain AI in 3 bullets to a child"
chat_with_llm(prompt, user_id)

In [ ]:
prompt = "Now do the same for Generative AI"
chat_with_llm(prompt, user_id)

In [ ]:
prompt = "Now do the same for machine learning"
chat_with_llm(prompt, user_id)

In [ ]:
prompt = "what topics have we discussed, show briefly as bullet points"
chat_with_llm(prompt, user_id)

## Alternative Implementation: RunnableWithMessageHistory with Full History

This is an alternative implementation demonstrating `RunnableWithMessageHistory` with:
1. Full conversation history (no windowing)
2. Custom session history factory
3. Cleaner separation of concerns


In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# Initialize LLM
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

# Create prompt template with history placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Be concise and informative."),
    MessagesPlaceholder(variable_name="history"),  # This will be populated with chat history
    ("human", "{input}")
])

# Create the base chain (prompt | llm)
base_chain = prompt | llm

# Session history factory function - creates/retrieves history for each session
def get_session_history(session_id: str) -> SQLChatMessageHistory:
    """
    Factory function that returns a SQLChatMessageHistory instance for the given session.
    Each session_id gets its own conversation history stored in SQLite.
    """
    return SQLChatMessageHistory(
        session_id=session_id,
        connection="sqlite:///chat_history_alt.db"  # Using a different DB file
    )

# Wrap the chain with message history management
chain_with_history = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",      # Key in input dict for the current message
    history_messages_key="history"   # Key in prompt for the message history
)

# Helper function to chat with a specific user session
def chat(user_input: str, session_id: str) -> str:
    """Send a message and get a response, maintaining conversation history per session."""
    response = chain_with_history.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    )
    return response.content

# Streaming version of the chat function
def chat_stream(user_input: str, session_id: str):
    """Send a message and stream the response."""
    for chunk in chain_with_history.stream(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    ):
        print(chunk.content, end="", flush=True)
    print()  # New line at the end


### Test with User 1 (Alice)


In [ ]:
# User 1: Alice asks about Python
print("Alice:", chat("My name is Alice. What are the best practices for Python programming?", "alice_session"))


### Test with User 2 (Bob) - Different Session


In [ ]:
# User 2: Bob asks about JavaScript (separate conversation history)
print("Bob:", chat("My name is Bob. What's new in JavaScript ES2024?", "bob_session"))


### Verify Memory Isolation - Each user remembers their own context


In [ ]:
# Alice asks a follow-up - should remember her name and previous topic (Python)
print("Alice follow-up:", chat("What's my name? And what were we discussing?", "alice_session"))


In [ ]:
# Bob asks a follow-up - should remember his name and previous topic (JavaScript)
print("Bob follow-up:", chat("What's my name? And what were we discussing?", "bob_session"))


### View Stored Conversation History


In [ ]:
# View Alice's conversation history stored in the database
alice_history = get_session_history("alice_session")
print("=== Alice's Conversation History ===")
for msg in alice_history.messages:
    role = "Human" if msg.type == "human" else "AI"
    print(f"{role}: {msg.content[:100]}...")

In [ ]:
# View Bob's conversation history stored in the database
bob_history = get_session_history("bob_session")
print("=== Bob's Conversation History ===")
for msg in bob_history.messages:
    role = "Human" if msg.type == "human" else "AI"
    print(f"{role}: {msg.content[:100]}...")

In [ ]:
prompt = "what topics have we discussed, show briefly as bullet points"
chat_with_llm(prompt, user_id)